# Очистка таблиц Users, Items и Users_Items

В этом ноутбуке я привожу таблицы в нормальный вид, оставляю только нужные столбцы и строки.

## Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import os
import re
import scipy.stats as sts
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from pathlib import Path

## Загрузка данных

In [2]:
BASE_PATH = os.path.abspath("../../../Tables/BaseTable")

In [ ]:
users_df = pd.read_csv(os.path.join(BASE_PATH, 'Users.csv'), sep=';', encoding='cp1251')
items_df = pd.read_csv(os.path.join(BASE_PATH, 'Items.csv'), sep=';', encoding='cp1251')
user_item_df = pd.read_csv(os.path.join(BASE_PATH, 'User_Items_test_month.csv'), sep=';', encoding='cp1251')
user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')

In [4]:
print("Изначальные размеры:")
print(f"Users: {users_df.shape}")
print(f"Items: {items_df.shape}")
print(f"User_Items: {user_item_df.shape}")
print(f"User_Item_year: {user_item_year_df.shape}")

Изначальные размеры:
Users: (2893, 18)
Items: (258, 8)
User_Items: (3133, 17)
User_Item_year: (40137, 1)


In [5]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 18 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Имя                                        2892 non-null   str    
 1   Телефон                                    2892 non-null   str    
 2   Email                                      997 non-null    str    
 3   Категории                                  58 non-null     str    
 4   Дата рождения                              2 non-null      str    
 5   Потратил, ?                                2892 non-null   str    
 6   Оплатил, ?                                 2892 non-null   str    
 7   Пол                                        6 non-null      str    
 8   Карта                                      0 non-null      float64
 9   Скидка                                     2892 non-null   float64
 10  Последний визит                    

In [6]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


In [7]:
user_item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3133 entries, 0 to 3132
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    1777 non-null   str    
 1   Специализация мастера           1777 non-null   str    
 2   Имя клиента                     1154 non-null   str    
 3   	Телефон                        1154 non-null   float64
 4   Email                           502 non-null    str    
 5   Время визита                    1777 non-null   str    
 6   Имя	 создателя                  1777 non-null   str    
 7   Дата создания                   1777 non-null   str    
 8   Статус                          1777 non-null   str    
 9   Источник                        1777 non-null   str    
 10  Категория услуги                3096 non-null   str    
 11  Название услуги                 3096 non-null   str    
 12  Стоимость, руб                  3096 non-null

In [8]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40137 entries, 0 to 40136
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                                                                              Non-Null Count  Dtype
---  ------                                                                                                                                                                                                                                                                              --------------  -----
 0   Имя	 мастера ,Специализация мастера ,Имя клиента ,	Телефон ,Email ,Время визита,Имя	 создателя,Дата создания ,Статус,Источник,Категория услуги,Название услуги ,"Стоимость, руб","Стоимость с учетом скидки, руб",Оплачено полностью,Комментарий,Категория,Unnamed: 17,Unnamed: 18  40137 non-null  

Заранее изучив данные, выявлены необходимые операции:

__Для Users__

1) Вместо Имя+Телефон+Email ввести `id_user`, а Email удалить
2) Удалить столбцы Дата рождения, Потратил в руб; Карта; Скидка; Последний визит; Чаевые; Количество посещений; Комментарий; Дополнительный телефон; Согласен на получение рассылок; Согласен на обработку персональных данных т.к. они большинством своим не информативны либо полностью пустые
3) В столбце Категория есть несколько возможных значений я хочу:
- все пропуски и значения "Клиенты отщепенцев", "ПРЕДОПЛАТА" заменить на "обычный"
- удалить все экземпляры (строчки), где категория "Черный список"
- заменить значение "Постоянный" на "обычный" (т.к. в таблице только 1 такой экземпляр)
4) Удалить пропуски, если они останутся

__Для Items__

1) Удалить все услуги, где значение "Категория товара или услуги в продаже" = "Солярий" (слишком много посещений и направление никак не связано с остальными услугами)
2) Удалить все услуги, которые были оказаны слишком маленькое количество раз ("Количество оказанных услуг"<8 раз) или где количество уникальных клиентов слишком маленькое ("Уникальные клиенты" < 7).
3) По сути каждая строчка это и есть информация об уникальной услуге, но я бы добавил столбец `id_item` и сделал его PK

__Для User-Items__

1) Удалить все элементы связанные с солярием
2) Заменить имена столбцов на отношение сущьностей (то есть столбец "Имя", который относиться к клиенту назвать "Имя клиента" и т.п.)
3) Удалить Создал (Имя, Дата);Статус; Источник; Информация (Статус, Имя, Дата); Оплачено полностью; Комментарий; Категория (они не имеют смысла или нулевые)
4) Добавить `user_id` (для каждого клиента такой же как в таблице Users(то есть чтобы одному клиенту и там и там значился 1 и тот же id)), добавить `item_id` (по аналогии с `user_id`) и удалить "Email"
5) Объеденить некоторые услуги исходя из бизнесс логики 

__Для User-Items-Year__

1) Повторить pipline для User-Items, но на файле с годом  


## Обработка Users

#### Вводим user_id для каждого уникального пользователя, определяя его с помощью тройки Имя+телефон+Email

In [9]:
def clean_phone(phone):
    if pd.isna(phone):
        return np.nan
    phone_str = str(phone).replace('.0', '').replace('+7', '8')
    return phone_str.strip()

In [10]:
users_df["Телефон"] = users_df["Телефон"].apply(clean_phone)
users_df = users_df.drop_duplicates(subset=["Телефон"], keep="first")
users_df["id_user"] = pd.Categorical(users_df["Телефон"]).codes

#### Настройка столбца 'Категории'

In [11]:
users_df['Категории'] = users_df['Категории'].fillna('обычный')
users_df['Категории'] = users_df['Категории'].replace({
    'Клиенты отщепенцев': 'обычный',
    'ПРЕДОПЛАТА': 'обычный',
    'Постоянный': 'обычный'
})

In [12]:
users_df['Категории'].value_counts()

Категории
обычный          2881
Черный список      12
Name: count, dtype: int64

In [13]:
users_df = users_df[users_df['Категории'] != 'Черный список'].copy()
users_df['Категории'].value_counts()

Категории
обычный    2881
Name: count, dtype: int64

Оставляем столбец, т.к. в будущем возможно введение других категорий.

In [14]:
users_df = users_df[[
    'id_user', 'Имя', 'Телефон',
    'Категории', 'Оплатил, ?', 'Последний визит'
]].copy()
users_df = users_df.rename(columns={'Оплатил, ?': 'Оплачено в руб'})

In [15]:
users_df.info()

<class 'pandas.DataFrame'>
Index: 2881 entries, 0 to 2892
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2881 non-null   int16
 1   Имя              2880 non-null   str  
 2   Телефон          2880 non-null   str  
 3   Категории        2881 non-null   str  
 4   Оплачено в руб   2880 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 140.7 KB


In [16]:
users_df.head()

,id_user,Имя,Телефон,Категории,Оплачено в руб,Последний визит
0,2706,Наталья,79859657657,обычный,50363,2026-03-04 13:00
1,462,дарья,79082451054,обычный,0,2026-03-04 13:00
2,965,Татьяна,79163219575,обычный,30880,2026-03-04 12:30
3,1924,Анастасия,79279021487,обычный,41470,2026-03-04 12:05
4,1206,Екатерина,79168230729,обычный,171450,2026-03-04 12:00


In [17]:
users_df[users_df['Телефон']=='79165161952']

,id_user,Имя,Телефон,Категории,Оплачено в руб,Последний визит
121,1050,Арина Фомичева,79165161952,обычный,58040,2026-02-28 14:05


#### Удалим пропуски и зафиксируем таблицу

In [18]:
users_clean = users_df.dropna(subset=['Последний визит']).copy()

In [19]:
users_clean.isna().sum()

id_user            0
Имя                0
Телефон            0
Категории          0
Оплачено в руб     0
Последний визит    0
dtype: int64

In [20]:
users_clean.info()

<class 'pandas.DataFrame'>
Index: 2711 entries, 0 to 2719
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2711 non-null   int16
 1   Имя              2711 non-null   str  
 2   Телефон          2711 non-null   str  
 3   Категории        2711 non-null   str  
 4   Оплачено в руб   2711 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 132.4 KB


#### Настроем типы данных

In [21]:
users_clean['Оплачено в руб'] = users_clean['Оплачено в руб'].astype(str).str.replace(',', '.').astype(float)

In [22]:
users_clean['Последний визит'] = pd.to_datetime(users_clean['Последний визит'], errors='coerce')

In [23]:
users_clean.dtypes

id_user                     int16
Имя                           str
Телефон                       str
Категории                     str
Оплачено в руб            float64
Последний визит    datetime64[us]
dtype: object

## Обработка Items

In [24]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


# Новая предобработка

#### Переименовываем столбцы 

In [25]:
items_df.columns = items_df.columns.str.strip()  # убираем пробелы в именах
items_df = items_df.rename(columns={
    "Выручка от продажи услуг, ?": "Выручка от продажи услуг, руб",
    "Ср. выручка с одного клиента, ?": "Ср. выручка с одного клиента, руб"
})

#### Настраиваем типизацию

In [26]:
def clean_money_col(series):
    return pd.to_numeric(series.str.replace(" ", "").str.replace(",", "."), errors="coerce")

def clean_percent_col(series):
    # убираем %, пробелы, заменяем запятые, приводим к числу
    return pd.to_numeric(
        series.str.replace(" ", "")
           .str.replace("%", "")
           .str.replace(",", "."),
        errors="coerce"
    )

items_df["Количество оказанных услуг"] = pd.to_numeric(
    items_df["Количество оказанных услуг"], errors="coerce"
).astype("Int64")

items_df["Доля от оказанных услуг в %"] = clean_percent_col(items_df["Доля от оказанных услуг в %"])
items_df["Выручка от продажи услуг, руб"] = clean_money_col(items_df["Выручка от продажи услуг, руб"])
items_df["% от общей выручки за услуги"] = clean_percent_col(items_df["% от общей выручки за услуги"])
items_df["Ср. выручка с одного клиента, руб"] = clean_money_col(items_df["Ср. выручка с одного клиента, руб"])
items_df["Уникальные клиенты"] = pd.to_numeric(
    items_df["Уникальные клиенты"], errors="coerce"
).astype("Int64")

In [27]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Название услуги или товара             258 non-null    str    
 1   Категория товара или услуги в продаже  258 non-null    str    
 2   Количество оказанных услуг             258 non-null    Int64  
 3   Доля от оказанных услуг в %            258 non-null    float64
 4   Выручка от продажи услуг, руб          258 non-null    float64
 5   % от общей выручки за услуги           258 non-null    float64
 6   Ср. выручка с одного клиента, руб      258 non-null    float64
 7   Уникальные клиенты                     258 non-null    Int64  
dtypes: Int64(2), float64(4), str(2)
memory usage: 16.8 KB


Удалчем услуги с солярием 

In [28]:
items_no_solar = items_df[
    items_df["Категория товара или услуги в продаже"] != "Солярий"
].copy()

In [29]:
items_filtered = items_no_solar[
    (items_no_solar["Количество оказанных услуг"] >= 6) &
    (items_no_solar["Уникальные клиенты"] >= 5)
].copy()

## Создаем items_dim

In [30]:
rename_map = {
    # брови
    "Коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "Коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "Укрепление акрилом": "Укрепление",
    "Укрепление гелем": "Укрепление",

    # покрытие
    "Укрепление цветным гелем": "Покрытие",
    "Покрытие гель лаком LUXIO, EMi, OneNail (руки)": "Покрытие",
    "Покрытие стойким лаком (ноги)": "Покрытие",
    "Покрытие база + топ (бесцветные)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (РУКИ)": "Покрытие",
    "Покрытие стойким лаком (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, EMi, OneNail (ноги)": "Покрытие",
    "Покрытие гель лаком EMi, OneNail (руки)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, E.Mi, OneNail (РУКИ) 1 НОГОТЬ": "Покрытие",
    "Покрытие гель лаком с эффектом \"Кошачий глаз\" (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, OneNail ФРЕНЧ (РУКИ)": "Покрытие",

    # снятие
    "Снятие лака (РУКИ)": "Снятие",
    "Снятие гель-лака (1 ноготь)": "Снятие",
    "Снятие гель-лака": "Снятие",
    "Снятие стойкого лака (руки)": "Снятие",
    "Снятие лака (НОГИ)": "Снятие",
    "Снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [31]:
ITEM_COL = "Название услуги или товара"

items_df[ITEM_COL] = items_df[ITEM_COL].replace(rename_map)

#### Добавим id_item

In [32]:
items_dim_cols = [
    "Название услуги или товара",
    "Категория товара или услуги в продаже"
]

# присвоим id_item (сквозной integer, как в твоей схеме)
items_dim = (items_filtered[items_dim_cols]
    .reset_index(drop=True)
    .reset_index()  # index = 0..N-1
    .rename(columns={"index": "id_item"})
)

#### Добавляем флаги комплексов 

In [33]:
items_dim["is_complex"] = 0          # 0 = базовая, 1 = комплекс
items_dim["complex_level"] = 0       # 0 = нет, 1 = мини, 2 = полный

In [34]:
items_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 5 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   id_item                                155 non-null    int64
 1   Название услуги или товара             155 non-null    str  
 2   Категория товара или услуги в продаже  155 non-null    str  
 3   is_complex                             155 non-null    int64
 4   complex_level                          155 non-null    int64
dtypes: int64(3), str(2)
memory usage: 6.2 KB


In [35]:
complex_names = [
    "Маникюр комплекс (полный)",
    "Маникюр комплекс (мини) 1",
    "Маникюр комплекс (мини) 2",
    "Педикюр комплекс (полный)",
    "Педикюр комплекс (мини) 1",
    "Педикюр комплекс (мини) 2",
    "Окрашивание и коррекция бровей (комплекс)",
    "Наращивание ресниц (комплекс)",
]

# категории этих комплексов (подставь свои, я возьму пример)
complex_cats = {
    "Маникюр комплекс (полный)": "Маникюр",
    "Маникюр комплекс (мини) 1": "Маникюр",
    "Маникюр комплекс (мини) 2": "Маникюр",
    "Педикюр комплекс (полный)": "Педикюр",
    "Педикюр комплекс (мини) 1": "Педикюр",
    "Педикюр комплекс (мини) 2": "Педикюр",
    "Окрашивание и коррекция бровей (комплекс)": "Брови",
    "Наращивание ресниц (комплекс)": "Ресницы",
}

In [36]:
# максимальный id_item, с которого продолжаем нумерацию
max_id = items_dim["id_item"].max()

new_complexes = []
for i, name in enumerate(complex_names):
    new_complexes.append({
        "id_item": max_id + i + 1,
        "Название услуги или товара": name,
        "Категория товара или услуги в продаже": complex_cats[name],
        "is_complex": 1,
        "complex_level": 2 if "комплекс (полный)" in name else 1,
    })

df_complexes = pd.DataFrame(new_complexes)

In [37]:
df_complexes.shape

(8, 5)

In [38]:
items_dim = pd.concat([items_dim, df_complexes], ignore_index=True)

In [39]:
items_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 5 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   id_item                                163 non-null    int64
 1   Название услуги или товара             163 non-null    str  
 2   Категория товара или услуги в продаже  163 non-null    str  
 3   is_complex                             163 non-null    int64
 4   complex_level                          163 non-null    int64
dtypes: int64(3), str(2)
memory usage: 6.5 KB


## Создание items_stats

In [40]:
items_stats_cols = [
    "id_item",
    "Название услуги или товара",
    "Категория товара или услуги в продаже",
    "Количество оказанных услуг",
    "Доля от оказанных услуг в %",
    "Выручка от продажи услуг, руб",
    "% от общей выручки за услуги",
    "Ср. выручка с одного клиента, руб",
    "Уникальные клиенты",
]

In [41]:
items_df["id_item"] = range(1, len(items_df) + 1)

items_stats = items_df[items_stats_cols].copy()

Упорядочиваем

In [42]:
items_stats = items_stats.sort_values("id_item").reset_index(drop=True)

In [43]:
items_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id_item                                258 non-null    int64  
 1   Название услуги или товара             258 non-null    str    
 2   Категория товара или услуги в продаже  258 non-null    str    
 3   Количество оказанных услуг             258 non-null    Int64  
 4   Доля от оказанных услуг в %            258 non-null    float64
 5   Выручка от продажи услуг, руб          258 non-null    float64
 6   % от общей выручки за услуги           258 non-null    float64
 7   Ср. выручка с одного клиента, руб      258 non-null    float64
 8   Уникальные клиенты                     258 non-null    Int64  
dtypes: Int64(2), float64(4), int64(1), str(2)
memory usage: 18.8 KB


## Обработка User-Items-Year

Удаляем пустые столбцы 

In [44]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40137 entries, 0 to 40136
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                                                                              Non-Null Count  Dtype
---  ------                                                                                                                                                                                                                                                                              --------------  -----
 0   Имя	 мастера ,Специализация мастера ,Имя клиента ,	Телефон ,Email ,Время визита,Имя	 создателя,Дата создания ,Статус,Источник,Категория услуги,Название услуги ,"Стоимость, руб","Стоимость с учетом скидки, руб",Оплачено полностью,Комментарий,Категория,Unnamed: 17,Unnamed: 18  40137 non-null  

In [45]:
user_item_year_df = user_item_year_df.loc[:, ~user_item_year_df.columns.str.contains("^Unnamed")]

Почистить имена колонок 

In [46]:
user_item_year_df.columns = user_item_year_df.columns.str.strip()

In [47]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40137 entries, 0 to 40136
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                                                                              Non-Null Count  Dtype
---  ------                                                                                                                                                                                                                                                                              --------------  -----
 0   Имя	 мастера ,Специализация мастера ,Имя клиента ,	Телефон ,Email ,Время визита,Имя	 создателя,Дата создания ,Статус,Источник,Категория услуги,Название услуги ,"Стоимость, руб","Стоимость с учетом скидки, руб",Оплачено полностью,Комментарий,Категория,Unnamed: 17,Unnamed: 18  40137 non-null  

In [48]:
COL_MASTER = "Имя мастера"
COL_MASTER_SPEC = "Специализация мастера"
COL_CLIENT_NAME = "Имя клиента"
COL_PHONE = "Телефон"
COL_EMAIL = "Email"
COL_VISIT_TIME = "Время визита"
COL_CATEGORY = "Категория услуги"
COL_SERVICE = "Название услуги"
COL_PRICE_DISCOUNT = "Стоимость с учетом скидки, руб"

In [49]:
def clean_money(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", ".", regex=False),
        errors="coerce"
    )

Приведение типов данных 

In [50]:
user_item_year_df[COL_PRICE_DISCOUNT] = clean_money(user_item_year_df[COL_PRICE_DISCOUNT])

KeyError: 'Стоимость с учетом скидки, руб'

Удалим строки без услуг (пропуски)

In [ ]:
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].notna()]
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].str.strip() != ""]

In [ ]:
user_item_year_df.shape[0]

39075

Фильтрация по солярию и абонимаентам 

In [ ]:
mask_not_solar = user_item_year_df[COL_CATEGORY] != "Солярий"
mask_not_sub = user_item_year_df[COL_CATEGORY] != "Услуги ПО АБОНЕМЕНТАМ"
user_item_year_df = user_item_year_df[mask_not_solar & mask_not_sub].copy()

In [ ]:
user_item_year_df.shape[0]

28857

Ренейм (объединение item)

In [ ]:
rename_map = {
    # брови
    "Коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "Коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "Укрепление акрилом": "Укрепление",
    "Укрепление гелем": "Укрепление",

    # покрытие
    "Укрепление цветным гелем": "Покрытие",
    "Покрытие гель лаком LUXIO, EMi, OneNail (руки)": "Покрытие",
    "Покрытие стойким лаком (ноги)": "Покрытие",
    "Покрытие база + топ (бесцветные)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (РУКИ)": "Покрытие",
    "Покрытие стойким лаком (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, EMi, OneNail (ноги)": "Покрытие",
    "Покрытие гель лаком EMi, OneNail (руки)": "Покрытие",
    "Покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, E.Mi, OneNail (РУКИ) 1 НОГОТЬ": "Покрытие",
    "Покрытие гель лаком с эффектом \"Кошачий глаз\" (руки)": "Покрытие",
    "Покрытие гель лаком Luxio, Beautix, OneNail ФРЕНЧ (РУКИ)": "Покрытие",

    # снятие
    "Снятие лака (РУКИ)": "Снятие",
    "Снятие гель-лака (1 ноготь)": "Снятие",
    "Снятие гель-лака": "Снятие",
    "Снятие стойкого лака (руки)": "Снятие",
    "Снятие лака (НОГИ)": "Снятие",
    "Снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [ ]:
user_item_year_df[COL_SERVICE] = user_item_year_df[COL_SERVICE].replace(rename_map)

In [ ]:
user_item_year_df.shape

(28857, 17)

Заполнение пропусков (полупстых строк)

In [ ]:
user_item_year_df = user_item_year_df.sort_values([COL_PHONE, COL_VISIT_TIME], ascending=False)

In [ ]:
fill_cols = [col for col in user_item_year_df.columns if col != COL_SERVICE]
user_item_year_df[fill_cols] = user_item_year_df[fill_cols].ffill(axis=0)

In [ ]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28857 entries, 13058 to 39286
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    28857 non-null  str    
 1   Специализация мастера           28857 non-null  str    
 2   Имя клиента                     28857 non-null  str    
 3   Телефон                         28857 non-null  object 
 4   Email                           28847 non-null  str    
 5   Время визита                    28857 non-null  str    
 6   Имя	 создателя                  28857 non-null  str    
 7   Дата создания                   28857 non-null  str    
 8   Статус                          28857 non-null  str    
 9   Источник                        28857 non-null  str    
 10  Категория услуги                28857 non-null  str    
 11  Название услуги                 28857 non-null  str    
 12  Стоимость, руб                  28857 non-nu

Добавляем id_user

In [ ]:
users_df["Телефон"] = users_df["Телефон"].astype(str).str.strip()
user_item_year_df[COL_PHONE] = user_item_year_df[COL_PHONE].astype(str).str.strip()

In [ ]:
user_item_year_df["Телефон"] = user_item_year_df["Телефон"].apply(clean_phone)

In [ ]:
user_item_year_df = user_item_year_df.merge(
    users_df[["id_user", "Телефон"]],
    on="Телефон",
    how="left"
)

In [ ]:
user_item_year_df.isna().sum()

Имя\t мастера                      0
Специализация мастера              0
Имя клиента                        0
Телефон                            0
Email                             10
Время визита                       0
Имя\t создателя                    0
Дата создания                      0
Статус                             0
Источник                           0
Категория услуги                   0
Название услуги                    0
Стоимость, руб                     0
Стоимость с учетом скидки, руб     0
Оплачено полностью                 0
Комментарий                       11
Категория                         10
id_user                           31
dtype: int64

Удаляем пропуски по id_user

In [ ]:
user_item_year_df = user_item_year_df[user_item_year_df["id_user"].notna() & user_item_year_df["Категория"].notna()].copy()
user_item_year_df["id_user"] = user_item_year_df["id_user"].astype("Int64")

In [ ]:
user_item_year_df.isna().sum()

Имя\t мастера                     0
Специализация мастера             0
Имя клиента                       0
Телефон                           0
Email                             0
Время визита                      0
Имя\t создателя                   0
Дата создания                     0
Статус                            0
Источник                          0
Категория услуги                  0
Название услуги                   0
Стоимость, руб                    0
Стоимость с учетом скидки, руб    0
Оплачено полностью                0
Комментарий                       1
Категория                         0
id_user                           0
dtype: int64

Рефакторинг по комплексам 

In [ ]:
MANICURE_SERVICES = [
    "Маникюр Комбинированный",
    "Маникюр Классический (обрезной)",
    "Маникюр Мужской аппаратный",
    "Маникюр Аппаратный",
    "Маникюр Мужской Классический",
    "Маникюр Европейский (необрезной)"
]

PEDICURE_SERVICES = [
    "Смарт Педикюр Мужской",
    "Педикюр Аппаратный",
    "Смарт Педикюр Женский",
    "Педикюр Классический",
    "Педикюр Мужской Комбинированный",
    "Педикюр Комбинированный"
]

LASH_EXT_SERVICES = [
    'Наращивание ресниц "1D"',
    'Наращивание ресниц «2,5D»',
    'Наращивание ресниц "1,5D"',
    'Наращивание ресниц "3D"',
    'Наращивание ресниц "2D"'
]

In [ ]:
def process_visit(df_visit: pd.DataFrame) -> pd.DataFrame:
    services = df_visit[COL_SERVICE].tolist()

    def has_any(service_list):
        return any(s in services for s in service_list)

    has_cover = "Покрытие" in services
    has_remove = "Снятие" in services
    has_align = "Выравнивание ногтевой пластины" in services
    has_manicure = has_any(MANICURE_SERVICES)
    has_pedicure = has_any(PEDICURE_SERVICES)
    has_brow_corr = "Коррекция бровей/форма бровей" in services
    has_brow_henna = "Окрашивание бровей хной" in services
    has_brow_paint = "Окрашивание бровей / ресниц (краской)" in services
    has_lash_ext = has_any(LASH_EXT_SERVICES)
    has_lash_remove = "Снятие поресничное" in services

    to_drop_idx = set()
    new_rows = []

    def get_idx(name, multiple=True):
        idxs = df_visit.index[df_visit[COL_SERVICE] == name].tolist()
        return idxs if multiple else (idxs[0] if idxs else None)

    def get_idx_any(names, multiple=True):
        idxs = []
        for n in names:
            idxs.extend(df_visit.index[df_visit[COL_SERVICE] == n].tolist())
        return idxs if multiple else (idxs[0] if idxs else None)

    # Брови, полный комплекс
    if has_brow_corr and (has_brow_henna or has_brow_paint):
        corr_idx = get_idx("Коррекция бровей/форма бровей", multiple=False)
        color_name = "Окрашивание бровей хной" if has_brow_henna else "Окрашивание бровей / ресниц (краской)"
        color_idx = get_idx(color_name, multiple=False)
        if corr_idx is not None and color_idx is not None:
            to_drop_idx.update([corr_idx, color_idx])
            row = df_visit.loc[corr_idx].copy()
            row[COL_SERVICE] = "Окрашивание и коррекция бровей (комплекс)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[corr_idx, color_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Маникюр: полный
    if has_cover and has_remove and has_align and has_manicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        remove_idx = get_idx("Снятие", multiple=False)
        align_idx = get_idx("Выравнивание ногтевой пластины", multiple=False)
        man_idx = get_idx_any(MANICURE_SERVICES, multiple=False)
        if all(idx is not None for idx in [cover_idx, remove_idx, align_idx, man_idx]):
            to_drop_idx.update([cover_idx, remove_idx, align_idx, man_idx])
            row = df_visit.loc[man_idx].copy()
            row[COL_SERVICE] = "Маникюр комплекс (полный)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, remove_idx, align_idx, man_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Маникюр: мини 1
    elif has_cover and has_manicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        man_idx = get_idx_any(MANICURE_SERVICES, multiple=False)
        if cover_idx is not None and man_idx is not None:
            to_drop_idx.update([cover_idx, man_idx])
            row = df_visit.loc[man_idx].copy()
            row[COL_SERVICE] = "Маникюр комплекс (мини) 1"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, man_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Маникюр: мини 2
    if "Маникюр комплекс (полный)" not in [r[COL_SERVICE] for r in new_rows]:
        if has_cover and has_remove and has_manicure:
            cover_idx = get_idx("Покрытие", multiple=False)
            remove_idx = get_idx("Снятие", multiple=False)
            man_idx = get_idx_any(MANICURE_SERVICES, multiple=False)
            if cover_idx is not None and remove_idx is not None and man_idx is not None:
                if not ({cover_idx, remove_idx, man_idx} & to_drop_idx):
                    to_drop_idx.update([cover_idx, remove_idx, man_idx])
                    row = df_visit.loc[man_idx].copy()
                    row[COL_SERVICE] = "Маникюр комплекс (мини) 2"
                    row["Стоимость с учетом скидки, руб"] = (
                        df_visit.loc[[cover_idx, remove_idx, man_idx], "Стоимость с учетом скидки, руб"]
                        .sum(skipna=True)
                    )
                    new_rows.append(row)

    # Педикюр: полный
    if has_cover and has_remove and has_align and has_pedicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        remove_idx = get_idx("Снятие", multiple=False)
        align_idx = get_idx("Выравнивание ногтевой пластины", multiple=False)
        ped_idx = get_idx_any(PEDICURE_SERVICES, multiple=False)
        if all(idx is not None for idx in [cover_idx, remove_idx, align_idx, ped_idx]):
            to_drop_idx.update([cover_idx, remove_idx, align_idx, ped_idx])
            row = df_visit.loc[ped_idx].copy()
            row[COL_SERVICE] = "Педикюр комплекс (полный)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, remove_idx, align_idx, ped_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Педикюр: мини 1
    elif has_cover and has_pedicure:
        cover_idx = get_idx("Покрытие", multiple=False)
        ped_idx = get_idx_any(PEDICURE_SERVICES, multiple=False)
        if cover_idx is not None and ped_idx is not None:
            to_drop_idx.update([cover_idx, ped_idx])
            row = df_visit.loc[ped_idx].copy()
            row[COL_SERVICE] = "Педикюр комплекс (мини) 1"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[cover_idx, ped_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    # Педикюр: мини 2
    if "Педикюр комплекс (полный)" not in [r[COL_SERVICE] for r in new_rows]:
        if has_cover and has_remove and has_pedicure:
            cover_idx = get_idx("Покрытие", multiple=False)
            remove_idx = get_idx("Снятие", multiple=False)
            ped_idx = get_idx_any(PEDICURE_SERVICES, multiple=False)
            if cover_idx is not None and remove_idx is not None and ped_idx is not None:
                if not ({cover_idx, remove_idx, ped_idx} & to_drop_idx):
                    to_drop_idx.update([cover_idx, remove_idx, ped_idx])
                    row = df_visit.loc[ped_idx].copy()
                    row[COL_SERVICE] = "Педикюр комплекс (мини) 2"
                    row["Стоимость с учетом скидки, руб"] = (
                        df_visit.loc[[cover_idx, remove_idx, ped_idx], "Стоимость с учетом скидки, руб"]
                        .sum(skipna=True)
                    )
                    new_rows.append(row)

    # Ресницы: полный комплекс
    if has_lash_ext and has_lash_remove:
        lash_idx = get_idx_any(LASH_EXT_SERVICES, multiple=False)
        rem_idx = get_idx("Снятие поресничное", multiple=False)
        if lash_idx is not None and rem_idx is not None:
            to_drop_idx.update([lash_idx, rem_idx])
            row = df_visit.loc[lash_idx].copy()
            row[COL_SERVICE] = "Наращивание ресниц (комплекс)"
            row["Стоимость с учетом скидки, руб"] = (
                df_visit.loc[[lash_idx, rem_idx], "Стоимость с учетом скидки, руб"]
                .sum(skipna=True)
            )
            new_rows.append(row)

    df_keep = df_visit.drop(index=list(to_drop_idx))
    if new_rows:
        df_new = pd.DataFrame(new_rows)
        return pd.concat([df_keep, df_new], ignore_index=True)
    return df_keep

In [ ]:
mask_complex = user_item_year_df["Название услуги"].str.contains(
    "комплекс", case=False, na=False
)

f"Всего строк с «комплекс» в названии услуг: {mask_complex.sum()}"

'Всего строк с «комплекс» в названии услуг: 219'

In [ ]:
if mask_complex.sum() > 0:
    print("\nУникальные названия комплексов:")
    print(
        user_item_year_df.loc[mask_complex, "Название услуги"].unique()
    )


Уникальные названия комплексов:
<StringArray>
[                                                                                                                                                'Чистка комплексная',
                                             'КОМПЛЕКС №2 (Ноги полностью (Голени, Колени, Бедра), Подмышечные впадины, Бикини на выбор (глубокое или классическое))',
                                                                              'КОМПЛЕКС №1 (Голени, Подмышечные впадины, Бикини на выбор (глубокое или классическое)',
 'КОМПЛЕКС №3 (Ноги полностью (Голени, Колени, Бедра), Руки полностью (Плечо, Предплечье, Локоть), Подмышечные впадины, Бикини на выбор (глубокое или классическое))']
Length: 4, dtype: str


In [ ]:
user_item_year_df[COL_VISIT_TIME] = pd.to_datetime(
    user_item_year_df[COL_VISIT_TIME],
    errors="coerce"
)

In [ ]:
group_cols = ["id_user", COL_VISIT_TIME]

processed_visits = []
for (uid, visit_time), df_visit in user_item_year_df.groupby(group_cols):
    processed = process_visit(df_visit)
    processed_visits.append(processed)

user_items_year_clean = pd.concat(processed_visits, ignore_index=True)

In [ ]:
mask_complex = user_items_year_clean["Название услуги"].str.contains(
    "комплекс", case=False, na=False
)

f"Всего строк с «комплекс» в названии услуг: {mask_complex.sum()}"

'Всего строк с «комплекс» в названии услуг: 245'

In [ ]:

print("\nУникальные названия комплексов:")
print(
    user_items_year_clean.loc[mask_complex, "Название услуги"].unique()
)


Уникальные названия комплексов:
<StringArray>
[                                                                             'КОМПЛЕКС №1 (Голени, Подмышечные впадины, Бикини на выбор (глубокое или классическое)',
                                                                                                                                                 'Чистка комплексная',
 'КОМПЛЕКС №3 (Ноги полностью (Голени, Колени, Бедра), Руки полностью (Плечо, Предплечье, Локоть), Подмышечные впадины, Бикини на выбор (глубокое или классическое))',
                                             'КОМПЛЕКС №2 (Ноги полностью (Голени, Колени, Бедра), Подмышечные впадины, Бикини на выбор (глубокое или классическое))',
                                                                                                                          'Окрашивание и коррекция бровей (комплекс)',
                                                                                                                      

In [ ]:
user_items_year_clean.shape

(28789, 18)

In [ ]:
user_items_year_clean[user_items_year_clean['Название услуги'] == 'Маникюр комплекс (мини) 1']

,Имя\t мастера,Специализация мастера,Имя клиента,Телефон,Email,Время визита,Имя\t создателя,Дата создания,Статус,Источник,Категория услуги,Название услуги,"Стоимость, руб","Стоимость с учетом скидки, руб",Оплачено полностью,Комментарий,Категория,id_user
17759,Эльмира,Мастер маникюра,Наталья,79037902515,yuliyamenshova@gmail.com,2025-07-17 18:00:00,Кудряшова Наталия Георгиевна,2025-07-14 11:07:43,Клиент пришёл,Web-интерфейс для администратора,Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник важен,292
17761,Лусине,Мастер маникюра,Наталья,79037902515,yuliyamenshova@gmail.com,2025-09-10 20:00:00,Рамазанова Мадина,2025-08-30 13:08:52,Клиент пришёл,Web-интерфейс для администратора,Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник важен,292
18193,Лиза,Мастер маникюра,Юлия,79057109144,panilena2013@gmail.com,2025-06-13 10:00:00,Рамазанова Мадина,2025-06-05 14:14:25,Клиент пришёл,Web-интерфейс для администратора,Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,ботокс\n\nс анестезией\nвремя не уменьшать,Сотрудник важен,370
19379,Элгиза,Мастер маникюра,Алина,79133834860,kochetkovaalina96@gmail.com,2025-05-18 20:00:00,Клиент,2025-05-13 22:54:22,Клиент пришёл,"""Онлайн-запись для сайта"" новый виджет через М...",Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,В 20.00 еще записана на педикюр. Возможно же б...,Сотрудник важен,657
20759,Айза,Мастер маникюра,Елена,79163217045,chimaroza_22@mail.ru,2025-06-30 14:00:00,Клиент,2025-06-25 08:02:24,Клиент пришёл,"""Онлайн-запись для сайта"" новый виджет через М...",Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,"не в 4 руки, это золовка клиентки",Сотрудник важен,964
20898,Гульнара,Мастер маникюра,Светлана,79163533649,Fivita@yandex.ru,2025-08-30 16:00:00,Клиент,2025-08-20 19:08:23,Клиент пришёл,"""Онлайн-запись для сайта"" новый виджет переход...",Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,добрый день! прошу вас поправить время на 18.10,Сотрудник важен,982
22612,Лиза,Мастер маникюра,Наталья,79197281940,nm7281940@yandex.ru,2026-03-01 12:00:00,Клиент,2026-02-22 17:01:33,Клиент пришёл,"""Форма компании"" новый виджет переход с zagar-...",Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2840.0,Да,хотела по возможности на руки к Дине,Сотрудник важен,1352
22657,Лиза,Мастер маникюра,Александра,79199603082,sasha_kyr@mail.ru,2026-01-21 10:00:00,Клиент,2026-01-18 21:02:35,Клиент пришёл,"""Форма компании"" новый виджет переход с zagar-...",Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2840.0,Да,"тут без снятия, поэтому полтора часа справится",Сотрудник важен,1362
23034,Гуля,Мастер маникюра,Мария и Дмитрий,79251084724,flash8805@bk.ru,2025-11-15 14:00:00,Клиент,2025-11-11 22:04:04,Клиент пришёл,Партнёры: Mobile app new widget через Мобильный,Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,"Просьба в 2руки\n\nтут будет стэмпиг леопард, ...",Сотрудник важен,1438
23062,Дина,Мастер маникюра,Ирина,79251227300,maria_sazykina@mail.ru,2025-06-30 12:00:00,Рамазанова Мадина,2025-06-26 16:02:22,Клиент пришёл,Web-интерфейс для администратора,Маникюр / Педикюр,Маникюр комплекс (мини) 1,990.0,2540.0,Да,по гарантии\nдлинные ногти время не ум\n\nМеди...,Сотрудник важен,1442


Добавление id_item

In [ ]:
user_items_year_clean["Название услуги_norm"] = (
    user_items_year_clean["Название услуги"]
    .str.replace(" ",  "", regex=False)
    .str.replace(" ", "", regex=False)  # неразрывный пробел
    .str.strip()
    .str.lower()
)

items_dim["Название услуги или товара_norm"] = (
    items_dim["Название услуги или товара"]
    .str.replace(" ",  "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
    .str.lower()
)

In [ ]:
user_items_year_clean = user_items_year_clean.merge(
    items_dim[["Название услуги или товара_norm", "id_item"]],
    left_on="Название услуги_norm",
    right_on="Название услуги или товара_norm",
    how="left"
)

# можно убрать временные колонки
user_items_year_clean = user_items_year_clean.drop(
    columns=["Название услуги_norm", "Название услуги или товара_norm"],
    errors="ignore"
)

In [ ]:
user_items_year_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28789 entries, 0 to 28788
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Имя	 мастера                    28789 non-null  str           
 1   Специализация мастера           28789 non-null  str           
 2   Имя клиента                     28789 non-null  str           
 3   Телефон                         28789 non-null  str           
 4   Email                           28789 non-null  str           
 5   Время визита                    28789 non-null  datetime64[us]
 6   Имя	 создателя                  28789 non-null  str           
 7   Дата создания                   28789 non-null  str           
 8   Статус                          28789 non-null  str           
 9   Источник                        28789 non-null  str           
 10  Категория услуги                28789 non-null  str           
 11  Название услу

In [ ]:
print("Пропусков id_item после нормализации:", user_items_year_clean["id_item"].isna().sum())

Пропусков id_item после нормализации: 11047


In [ ]:
print("Пропусков id_item:", user_items_year_clean["id_item"].isna().sum())

Пропусков id_item: 11047


In [ ]:
# 1) номера строк с пропуском
mask_nan = user_items_year_clean["id_item"].isna()

# 2) уникальные названия, которые не нашлись в items_dim
missing_services = user_items_year_clean.loc[mask_nan, "Название услуги"].unique()
print("Уникальные названия, для которых нет id_item (первые 50):")
print(missing_services[:50])

Уникальные названия, для которых нет id_item (первые 50):
<StringArray>
[                                                                      'Снятие',
                                                                     'Покрытие',
                              'Удаление волос бритвой перед лазерной эпиляцией',
                                                                   'Укрепление',
                                                      'Стразы сваровски (1 шт)',
                                                'Сахарная эпиляция спина/живот',
                                             'Дизайн "Слайдер" (заливка ногтя)',
                                                'Коррекция бровей/форма бровей',
                                        'Лазерная Эпиляция (FOR WOMEN) - Спина',
                         'Покрытие гель лаком с эффектом "Кошачий глаз" (ноги)',
                                           'Рисунок дотсами простой (1 ноготь)',
                                     

In [ ]:
print("Снятие:")
print(items_dim["Название услуги или товара"].str.contains("Снятие", case=False).sum())

print("Покрытие:")
print(items_dim["Название услуги или товара"].str.contains("Покрытие", case=False).sum())

print("Коррекция бровей:")
print(items_dim["Название услуги или товара"].str.contains("Коррекция бровей", case=False).sum())

print("Комплексы:")
print(
    items_dim["Название услуги или товара"]
    [items_dim["Название услуги или товара"].str.contains("комплекс", case=False)]
    .unique()
)

Снятие:
8
Покрытие:
13
Коррекция бровей:
3
Комплексы:
<StringArray>
['КОМПЛЕКС №3 (Ноги полностью (Голени, Колени, Бедра), Руки полностью (Плечо, Предплечье, Локоть), Подмышечные впадины, Бикини на выбор (глубокое или классическое))',
                                             'КОМПЛЕКС №2 (Ноги полностью (Голени, Колени, Бедра), Подмышечные впадины, Бикини на выбор (глубокое или классическое))',
                                                                              'КОМПЛЕКС №1 (Голени, Подмышечные впадины, Бикини на выбор (глубокое или классическое)',
                                                                                                                                                 'Чистка комплексная',
                                                                                                                                          'Маникюр комплекс (полный)',
                                                                                                 

In [ ]:
user_items_year_clean[user_items_year_clean['id_user'] == 1050].sort_values(by='Время визита', ascending=False)

,Имя\t мастера,Специализация мастера,Имя клиента,Телефон,Email,Время визита,Имя\t создателя,Дата создания,Статус,Источник,Категория услуги,Название услуги,"Стоимость, руб","Стоимость с учетом скидки, руб",Оплачено полностью,Комментарий,Категория,id_user,id_item
21230,Лусине,Мастер маникюра,Арина Фомичева,79165161952,sv.diker@gmail.com,2026-02-28 14:05:00,Клиент,2026-02-25 17:02:38,Клиент пришёл,"""Форма компании"" новый виджет переход с zagar-...",Маникюр / Педикюр,Маникюр Аппаратный,990.0,990.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник не важен,1050,96.0
21229,Лусине,Мастер маникюра,Арина Фомичева,79165161952,sv.diker@gmail.com,2026-01-28 12:00:00,Наталья,2026-01-22 22:56:23,Клиент пришёл,Мобильное приложение для администратора,Маникюр / Педикюр,Снятие,490.0,490.0,Да,взять с нее 2 500 - остальное записать в расхо...,Сотрудник важен,1050,NaN
21228,Мария,Мастер маникюра,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-11-26 12:00:00,Наталья,2025-11-21 19:48:55,Клиент пришёл,Web-интерфейс для администратора,Маникюр / Педикюр,Маникюр Комбинированный,990.0,990.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник важен,1050,14.0
21227,Мария,Мастер маникюра,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-10-25 12:00:00,Наталья,2025-10-17 17:48:32,Клиент пришёл,Мобильное приложение для администратора,Маникюр / Педикюр,Маникюр Аппаратный,990.0,990.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник важен,1050,96.0
21226,Лусине,Мастер маникюра,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-09-26 12:00:00,Клиент,2025-09-26 01:31:29,Клиент пришёл,"""Форма компании"" новый виджет переход с zagar-...",Маникюр / Педикюр,Маникюр Аппаратный,990.0,990.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник не важен,1050,96.0
21225,Лусине,Мастер маникюра,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-08-24 10:00:00,Клиент,2025-08-21 18:50:11,Клиент пришёл,"""Форма компании"" новый виджет переход с zagar-...",Маникюр / Педикюр,Маникюр Комбинированный,990.0,990.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник не важен,1050,14.0
21224,Дина,Мастер маникюра,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-07-31 12:00:00,Наталья,2025-07-30 18:46:36,Клиент пришёл,Мобильное приложение для администратора,Маникюр / Педикюр,Маникюр Комбинированный,990.0,990.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник не важен,1050,14.0
21223,Айка,Мастер по наращиванию ресниц,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-06-20 14:00:00,Наталья,2025-06-17 22:12:50,Клиент пришёл,Мобильное приложение для администратора,Наращивание ресниц,Снятие поресничное,450.0,450.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник важен,1050,122.0
21222,Айка,Мастер по наращиванию ресниц,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-05-18 12:00:00,Наталья,2025-05-15 08:03:40,Клиент пришёл,Мобильное приложение для администратора,Наращивание ресниц,"Наращивание ресниц ""2D""",2700.0,2700.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник не важен,1050,115.0
21221,Айка,Мастер по наращиванию ресниц,Арина Фомичева,79165161952,sv.diker@gmail.com,2025-04-22 12:30:00,Наталья,2025-04-20 00:00:08,Клиент пришёл,Мобильное приложение для администратора,Наращивание ресниц,"Наращивание ресниц ""2D""",2700.0,2700.0,Да,Запись создана через “Яндекс.Карты“.\nПожалуйс...,Сотрудник не важен,1050,115.0


======================================================

## Сортировка таблиц по id

In [ ]:
# 1. users_clean по user_id
users_clean = users_clean.sort_values('id_user').reset_index(drop=True)

# 2. items_clean по item_id
items_clean = items_clean.sort_values('id_item').reset_index(drop=True)

# 3. user_items_final по user_id
user_item_final = user_items_final.sort_values('user_id').reset_index(drop=True)

NameError: name 'items_clean' is not defined

## Выгрузка таблиц

In [ ]:

''' 
users_clean.to_csv('users_clean_final.csv', index=False, encoding='utf-8-sig')
items_clean.to_csv('items_clean_final.csv', index=False, encoding='utf-8-sig')
user_item_final.to_csv('user_items_final.csv', index=False, encoding='utf-8-sig')
'''